# Fine-tuning de BERT para clasificar problemas de matemáticas

### Notebook 2 de 4 · Arquitectura encoder-only (clasificación)

**Proyecto:** Tutor inteligente de matemáticas · Comparación de arquitecturas
mediante fine-tuning
**Curso:** SI4006 · Tópicos Especiales y Aplicaciones en IA · Universidad EAFIT
**Notebook base:** `S04_Lab_Fine_tuning_Qwen.ipynb`

---

## 1 · Introducción

### Qué problema resolvemos aquí (y cuál no)

**BERT no genera soluciones.** Es importante decirlo desde el principio porque
es el malentendido más común del laboratorio. Un modelo encoder-only no tiene
cabeza de lenguaje autoregresiva: no puede escribir "Paso 1: dividimos 48 entre
6". Lo que hace es *entender* el texto y producir una representación densa que
alimenta un clasificador.

Su papel en el tutor es el de **enrutador**: dada la pregunta de un estudiante,
decidir de qué tipo de problema se trata. Esa decisión es la que el notebook 3
usará para especializar la generación.

### Las 11 categorías

| Categoría | Ejemplo |
|---|---|
| `suma` | "Una biblioteca tenía 145 libros y recibió 89 más" |
| `resta` | "Un tanque contiene 250 litros y se usan 88" |
| `multiplicacion` | "15 cajas con 24 lápices cada una" |
| `division` | "Repartir 48 caramelos entre 6 amigos" |
| `operaciones_combinadas` | "(45 - 9) ÷ 6 + 11" |
| `potencias_raices` | "√144 + 8" |
| `fracciones` | "5/6 + 1/3" |
| `porcentajes` | "El 25% de 420 estudiantes" |
| `ecuaciones` | "2x + 5 = 19" |
| `geometria` | "Área de un círculo de radio 5" |
| `estadistica_probabilidad` | "Probabilidad de sacar un 4 en un dado" |

### El manual de etiquetado

Las categorías se solapan por naturaleza: "3.75 + 2.48" es una suma *y* una
operación con decimales; "el 25% de 420" es un porcentaje *y* una
multiplicación. Sin una regla explícita, el etiquetado sería inconsistente y el
modelo aprendería ruido.

La regla que se aplicó al construir el corpus es una **cascada de precedencia**,
del criterio más específico al más general:

```
1. ¿Pregunta por área, perímetro, ángulos, volumen?  -> geometria
2. ¿Hay que despejar una incógnita?                  -> ecuaciones
3. ¿Es promedio, mediana, moda, rango, probabilidad? -> estadistica_probabilidad
4. ¿Interviene un porcentaje?                        -> porcentajes
5. ¿Los operandos son fracciones?                    -> fracciones
6. ¿Hay potencias o raíces?                          -> potencias_raices
7. ¿Dos o más operaciones distintas / paréntesis?    -> operaciones_combinadas
8. En otro caso, la única operación presente         -> suma|resta|multiplicacion|division
```

Consecuencia deliberada: "área de un cuadrado de lado 12" es `geometria`
aunque implique una potencia, y "3.75 + 2.48" es `suma` porque los decimales
son un formato numérico, no una destreza distinta.

### Ventajas y limitaciones del encoder-only

**Ventajas**
- Atención **bidireccional**: cada token ve el contexto completo, a izquierda y
  derecha. Para entender de qué trata una frase, eso es estrictamente mejor que
  la atención causal de un decoder.
- Muy eficiente: una sola pasada hacia adelante, sin generación token a token.
  La inferencia es dos órdenes de magnitud más rápida que la de Qwen.
- 110M de parámetros: el fine-tuning completo cabe sin problema en una T4.
- La salida es una distribución sobre 11 clases, así que se puede medir con
  métricas cerradas y bien entendidas (F1, matriz de confusión).

**Limitaciones**
- No genera texto. No sirve como tutor por sí solo.
- El conjunto de categorías es cerrado: una pregunta de trigonometría se
  clasificará forzosamente como una de las 11 clases existentes.
- Con 12 ejemplos de entrenamiento por clase, es fácil que aprenda atajos
  léxicos (ver la palabra "porcentaje") en vez de la estructura del problema.

## 2 · Objetivos

1. Establecer **tres baselines** de dificultad creciente antes de tocar BERT:
   clase mayoritaria, TF-IDF + regresión logística, y BERT con la cabeza sin
   entrenar.
2. Hacer fine-tuning de BETO (BERT en español) para clasificación en 11 clases.
3. Evaluar con accuracy, precision, recall y F1 (macro y ponderado), más el
   reporte por clase y la matriz de confusión.
4. Identificar **qué categorías se confunden entre sí** y explicar por qué.
   Esa información es la que determina si el pipeline del notebook 3 es viable.
5. Registrar todo en W&B y en `resultados/bert.json`.

## 0 · Preparación del entorno

Instalamos el ecosistema Hugging Face. Las versiones se fijan por rango mayor
para evitar que un cambio de API rompa el notebook meses después.

> **Antes de empezar:** activen la GPU en `Entorno de ejecución → Cambiar tipo
> de entorno de ejecución → T4 GPU`. Sin GPU el entrenamiento es inviable.

**Sobre la desinstalación de `torchao`.** Colab trae `torchao` preinstalado, y
`peft` comprueba su versión con una función que **lanza `ImportError` en lugar
de devolver `False`** cuando la encuentra más antigua de lo que espera. El
resultado es que `get_peft_model()` falla con un error que no tiene ninguna
relación aparente con LoRA.

Ningún notebook del proyecto usa cuantización de torchao, así que lo quitamos.
La alternativa —actualizarlo— también funcionaría, pero torchao está acoplado a
la versión de torch y actualizarlo puede arrastrar un torch distinto y romper
otras cosas en Colab. Desinstalarlo no afecta a nada de lo que hacemos aquí.

In [ ]:
%pip install -q "transformers>=4.44" "datasets>=2.20" "peft>=0.12" \
    "accelerate>=0.33" "bitsandbytes>=0.43" "scikit-learn>=1.3" wandb

# Ver la nota de arriba: evita que get_peft_model() falle con un ImportError
# de torchao que nada tiene que ver con LoRA.
%pip uninstall -y -q torchao

print("Librerías instaladas. Si Colab pide reiniciar la sesión, reinícienla y sigan desde aquí.")

In [ ]:
import os, random, sys
import numpy as np
import torch
import transformers

SEMILLA = 42
random.seed(SEMILLA)
np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)
transformers.set_seed(SEMILLA)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"transformers  {transformers.__version__}")
print(f"torch         {torch.__version__}")
print(f"Dispositivo   {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU           {torch.cuda.get_device_name(0)}")
else:
    print("AVISO: sin GPU el fine-tuning tardará horas. Activen el runtime T4.")

# Detección temprana del conflicto peft/torchao. Más vale que salte aquí, en la
# celda de entorno, que dentro de get_peft_model() veinte celdas más adelante
# con un mensaje que no menciona LoRA por ninguna parte.
try:
    from peft.import_utils import is_torchao_available
except Exception:
    pass                       # ruta interna de peft cambiada: no es un problema
else:
    try:
        is_torchao_available()
    except ImportError as e:
        print(f"\nAVISO peft/torchao: {e}")
        print("  Solución: ejecuten  %pip uninstall -y torchao  y reinicien la sesión.")

### Persistencia entre notebooks

Los notebooks 3 y 5 **leen carpetas que producen los notebooks 1, 2 y 4**:

```
1 · Qwen    -> adaptadores/qwen-lora/      ─┐
2 · BERT    -> modelos/bert-clasificador/  ─┤-> el notebook 3 las lee
4 · FLAN-T5 -> adaptadores/flan-t5-lora/    │
1,2,3,4     -> resultados/*.json           ─┴-> el notebook 5 los lee
```

En Colab, `/content` se borra al desconectar el runtime, así que ese trabajo se
perdería entre sesiones. Montando Google Drive y trabajando desde una carpeta
suya, los artefactos sobreviven y cada notebook se puede ejecutar el día que se
pueda.

Con `USAR_DRIVE = False` todo queda en `/content`, lo cual es válido si
ejecutan los notebooks 1, 2 y 3 seguidos sin desconectar.

Fuera de Colab la celda no hace nada: el directorio de trabajo se queda como
está.

> **Los checkpoints intermedios nunca van a Drive.** El `Trainer` guarda en
> `output_dir` el modelo *más el estado del optimizador* en cada época. Para el
> notebook 2, que hace fine-tuning completo de BETO, eso son varios GB que
> además se escribirían por red. Como son desechables —lo que importa es el
> modelo final—, se mandan siempre al disco local del runtime mediante
> `DIR_CHECKPOINTS`.

In [ ]:
import os
from pathlib import Path

USAR_DRIVE    = True
CARPETA_DRIVE = "/content/drive/MyDrive/ProyectoIA"

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB and USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CARPETA_DRIVE).mkdir(parents=True, exist_ok=True)
    os.chdir(CARPETA_DRIVE)

# Checkpoints del Trainer: grandes y desechables -> siempre en disco local.
DIR_CHECKPOINTS = "/content/salidas" if EN_COLAB else "salidas"

print(f"En Colab           : {EN_COLAB}")
print(f"Directorio de trabajo: {Path.cwd()}")
print(f"Checkpoints en     : {DIR_CHECKPOINTS}")

## 3 · Arquitectura del modelo

```
              "¿Cuál es el 25% de 420 estudiantes?"
                              |
                     tokenización WordPiece
                              |
        [CLS]  ¿ cuál es el 25 % de 420 estudiantes ?  [SEP]
                              |
                    +---------v----------+
                    |  Embeddings        |
                    | (token+pos+segm.)  |
                    +---------v----------+
                              |
                   +----------v-----------+
                   |  12 bloques BERT:    |
                   |  - Autoatención      |  <- BIDIRECCIONAL: cada token
                   |    (12 cabezas)      |     ve toda la frase
                   |  - FFN + LayerNorm   |
                   +----------v-----------+
                              |
                  vector [CLS] de 768 dimensiones
                              |
                    +---------v----------+
                    | Cabeza lineal 768→11|  <- esto es lo NUEVO,
                    +---------v----------+      inicializado al azar
                              |
                   logits sobre 11 categorías
```

La diferencia decisiva frente a Qwen está en la máscara de atención: BERT no
tiene ninguna. El token `25` puede atender a `estudiantes`, que está a su
derecha. Un decoder no puede hacerlo, y por eso los encoders siguen siendo la
mejor opción para clasificar.

El vector `[CLS]` es un resumen aprendido de la secuencia completa. Sobre él se
monta una capa lineal de 768×11 que es la única parte del modelo que no viene
preentrenada.

In [ ]:
MODELO_ID = "dccuchile/bert-base-spanish-wwm-cased"   # BETO: BERT entrenado en español
# Alternativa multilingüe (peor en español, útil si BETO no está disponible):
# MODELO_ID = "bert-base-multilingual-cased"

DIR_MODELO   = "modelos/bert-clasificador"
LONGITUD_MAX = 64        # se justifica en la sección 6

> **Por qué BETO y no `bert-base-multilingual-cased`.** mBERT reparte un
> vocabulario de 119k tokens entre 104 idiomas; BETO dedica sus 31k tokens
> exclusivamente al español. Sobre texto en español, BETO parte las palabras en
> menos piezas y con cortes más lingüísticos ("estudiantes" como una unidad en
> lugar de "estu ##dian ##tes"). Con 132 ejemplos de entrenamiento, esa
> eficiencia de representación importa mucho más que en un corpus grande.

## 4 · Carga del dataset

### 4.1 · Materialización del corpus

El corpus vive en `data/math_tutor_dataset.jsonl`, generado por
`scripts/dataset_fuente.py`. Para que el notebook funcione en Colab sin subir
archivos, abajo va una **copia comprimida** de ese mismo archivo.

La celda **no sobrescribe** el JSONL si ya existe: eso permite escalar el
dataset (reemplazar el archivo por uno mayor) sin tocar el notebook. El hash
SHA-256 que se imprime debe ser idéntico en los cinco notebooks; si difiere,
alguno está entrenando con datos distintos y la comparación no sería válida.

Hash esperado de la versión embebida: `cf4732834d64a196…`

In [ ]:
import base64, gzip, hashlib, json
from pathlib import Path

RUTA_DATOS = Path("data/math_tutor_dataset.jsonl")
SHA_ESPERADO = "cf4732834d64a196d49eda88ddac8694a529a8bfc6b843e8eb8d74b8c6d59ecc"

_BLOB = (
    "H4sIAAAAAAAC/9V9XY/jSJLYuwH/B2KABrrR3ZL4KaqAQWM8O4DvMLc3t+Pzi20sWBK7mmNJrKWkQlUfDPje7h8s4Ld5nId5"
    "2BsbBublgKkf4P+wv8QRkUkyM5lBJlWqKhWw21OimBmhjMiMz4z4py+K1RcX3he7w+b9zP/iHfx1vS72+GhfZcUWnyyzfX5V"
    "VkUmX8y0h3+kCWbwaF9cl/hKeZ1X2bIoafC2uMnX+PQy2xXLEh/BqKt8K5/l+CTfArAVzf+HfHfI1ze5t868XXF1KOC73JNT"
    "3v+yvfCCKPbeeuE8nRC62boQI7/LdqXnX3jfA4abcucdtvDFKt+981b5Mt9mO+/OW+Js+CeM2ubZdlUCnJ23LD5W2W7yX7c0"
    "R6DA8L70kiCEbwCv60O+22fex2KbrS/wMcK/hhE7AP9fvjgeLs5jQvziv8HDqgb6RwKKv1KCvcnWZaV8znd/XOUbXP6P2XqX"
    "/49//+/+SaVs8CIoG07muAjBJOJo+xUtH64yTLIsNxmucbHJ1l4Gk+3gk4c/o1JIqUyKSzthqDkxyekMCodZgLAEnJgUnDiR"
    "MDwNCa+r8nKdi9fGUvBvD9nWq/JlWVVAMFjZ2Pvvxfr+l02+r2CdrssK12qT3f8l22bA9cEktb2wz6pVPvF++7evD/c/bvfw"
    "hfpSO3++9fblPlt/sPPCN2vxtZcTifC3I1FwX61gnlUBy75dFtq+RpTfEl5IJYYTQhUfnSncYQqu0KHxLNEBqXCHC3NET72/"
    "t4f8phy9wf0gnuEpF6WzgcN7XYoV3d7/uslhTRQaKpPAukbzcGYjIz63n8+dqfE1y6QcteqZGxLVDwZoFL8MGs3EUTabJAyJ"
    "vtktq+KywKWEl/BoLPGPmXedVZkHc2brrBKrLA7MXKVeOz0utD8JYxv18Lmx846AijNY4HGErYE2hK0fDBA2eSmbDw8j0jJS"
    "x913XRW0RdrBX4KKolCzHpDDuZhXS3gXztg5gYBX51bazuP+fckAxUGD4DjSznXCzl3IOn9igWuj6j+CHL0sLtdFuc+XIDnz"
    "7f3PmeeDrrguLlFa3qG8hH0B4jJd1A9ppp0mYeU3e2CL3Ms+lVX2gdOw5IyVPhutOYm+7LABDLN3Xra7/9n700HoQRv9jCZt"
    "FjD60gvCyMYF8FgCMNWu4+HTQa5D5hhCB98wBjx24IxU44wbWsO93LhPyh5fZ5sC9jvSswIO8OPZDA7EfCeWa33Y5sghAcg2"
    "5THorKC/KtzRjB9WuOjN0q4Dye27Ag7VWIGQeiuxwK0M/7VxBH0h0OwoXc5wiQWsEDlOMMA2rEDPHZhhcS7M8BVYnh7+tlVR"
    "evS8Kreen4Jik60O6z2dFsDgM29b3P9FPR9AewU5sSvRQG0GDjDD3x5gnHpsX1WHa5K+zVwqE6SkXhF0UK/8hV1ng+fNaJ0H"
    "XMAR7buAWD3OhNYqdPDNMOX92WPK/QLEd7XJgZankP3RbJHAsiyC9Hi9u50DVjaeBdZjHZ+PULu7c3LUqiduiFQ/GCCSfyZS"
    "vDrs6bja5dtVXhU7UF+FMIZNKzQeD96Hp/BOMAl0y/oO39rlVwd0HDWvkdNBea3e0PUJuS63V8X+sJLHJtmphMeHAR5QhioH"
    "LMHVnGQToU0iGmjg2oxp3pR2g0auMQMOa0bzRrQDowQvZzf76WQBCzKfzOJhG41eJnMJ/3K00sSrAgRqUvFkYdXl6QvWUBsB"
    "WpzdFqCsAldDbnW3+skAocMzORFw+68y7wb+RfU9mntLUOd2OYpjUNN29z9eZrDd77wkxM+rclNsr0pNarcD6llWhwrdkzgA"
    "lkocOBvwxg1ueZoCZ+pR5SI0tQAbsKN9qwCHxy1O3HbnIZFE0GCwproJqLXZfRfRHZ1QaXuKHT9PyawJ/Dg+XoArk+DygpJs"
    "JyJ+MUKG26Zl6VbP3dKrfjJAsfgMdu03W5DDYJCXq/wq8z5ldyClctyFPwhuho32OUMleomKdwBwFzHu31kTE9J2b/M6zjSg"
    "bqvbp3WSZBI4ukrw5FzQuenHFg8N+t3JZZLRkAt8DXcaqsmB3VEDjxsc+c3sgI3quhnCgz3vDWTaY3/ImQOTuYU2K4Ro4R//"
    "6SJgKbi83ntRvGB2+R8QRVxGWxwRWO26WKFUKYEiZAmCnliCvgkcBgwLC7PPFNZogKGNHFuDIfBYp/wJEMAJTdCscR7rsTL5"
    "eYDYwQsh9iSCRQgnwXxYi8N3SZOCPxx1OHpTzI/G1cS3UhifswqcM1SiaRcea8xJoK0xJx8MEDZ0Fdsu1LWKAk5sWwj8LZ5k"
    "G6TnHpQuD080YPQEPBMZnolf5+tdcbBERUEMLKW2tqhfrtU2eHFbLj/lrSn38ZCLM7OFRMvIyIl/3Mr54eMuB1cq7kNUtzxa"
    "EtiZCockQC7cge/Dd14uyFm9EzJN4nWZ/VB66Ge3sc770PitOh85oIIDxiDB8ZMVk4a53ruwVvTkZ8ZxmiD5Nt+DhJ3HQzKi"
    "RxVUZiEHaRDbPbJBzBz/Vk3QMivvdQ1iw90auMjx+IWQyaeVgLCVi4UO7worGTRnVwOdXhUQKGNoEtuTlCY95rkzXLLOLRD5"
    "JKVJbGQpTVyom5yIuieNuZGXLgSNtgm5Cd0Gg1y2GBtEpFaQJwO5INflFubJd8xZ/W07ptaWdnhS4ql5wIMSD1AZ9VJzWALa"
    "ZBFZW3bNHR5bg2ujIFIGiwGLtes0gK1x56Sbz8+D6pCXtMVoIhhvguaYEgJIVSLMkMHOANrDah32xboAQ8RLU/m9wQb7lg0+"
    "DJzRsNo0sZwT/EHgqUUMcrAYS6/YFpBNpFIfkXqPkIEgsKtsxE8CiQNzdjvCJNerDo0lvwayJX8SOJA/fdQj/cSe15kQcUn4"
    "IPGrToOSMmHkbzJO/tqmZQVwYgrgxEkAL85hs361XVX3P+3qfIhIiXvfeVfZbg+6dZA2cV011r0CsxUiyetcbFA+CwKngfdK"
    "oa1u6BBY18O7+zKSyo8Ai9uFCXPTF7YwtytM8pJaofHb0x7ipufDNDcCnQ80u57EXTpB98JskjpoX/guaUHwh6PyRW+K+UVm"
    "Z8wkkvK6lzNU8rB24fWkksZm/qjDtvb9x9zWY4j7ffkRt/SmAJHrT5LUa0Kgu4P3CabJtmX9bRx49sjnqviYVzkm4qLkRrSF"
    "iQ5gMDQCK8ypZNpYkPXlpdAFapMVZ2/mgzwkMO4zYd7flWrWOaH+XuD4JVDOt4ZH8blnC44+EA8SBl0MOKYx0Gh4B5878E7w"
    "QmyyZIb+NvbeiIvgrqdAB1dgJSk8HiOyjQlZh5mYtfWXBS6EeXx32RjtOs/2VZ30gPIIhuZbkZUExj9yM0hor1wertEs0VTq"
    "5lVpW6GpwZpV2oI3I+t5Udc1LapYGuu4UXxIkbFJbchQqefqoW8POOkg0QCxAtuA1krsIHWge/SCNOr3MXmMmD35HeQqFkIP"
    "wjcx2xWYaJnjr99hfnAdWMP4GWa35Eug2fawuf+pKpZqsEPCQUejP7d6M/25TtiHwsbZdKis+1KAbh2W4vMAleNzEdkQFv0P"
    "5VUJ29f0VItrUn5k+sQp3Ama7e/zGwoRwZ9/OBSf5fsdb26/eFcAOkn1fGeiSUKTpCh9ReJUS2YGIr5+H77BXRthJhJ5Q+yO"
    "l16X+FGIiKxmOwq8P6bPIT7EXpvD2i1iCi8CF62B25emWBF8Fjzh5cHEu/+zhxmEVh74Hbi4yg145HD3tm+TmUx/0vUf+jNS"
    "VX167kf4YppYUx7hsU7lUaBIvzeAcFSVkBo6ys8DhAyehpAjImcOqhr4mGF5/AknGP6uQRgPY7KkdgXEzrxPh8uCkgvgxiYs"
    "fiLmwRN4ns4Uwn6D9wx2MAseQEIpwe0iA9mN/YXGHl0Jg/9YL7NMUp34RyKGkxyDEnvhReDV3niZpA6sEp6aVR4aYwW9MTvs"
    "y839LzfF2rspIGME7BxMwoG5yiXKX/Rcok20Rzc27NfOjVO87DJw2zSvQHaBSAroZdJIZ2BkbgE0p2V+XW5vcqkgoMu8gGMc"
    "blKICS6sM2GGC/A1faFw4u/qS6PwffurgDnknF/ij8Jjg+Iufmr3KgFHsKm2D8UVZxuPJSubuqi2kimdObCpc/bew3nVUTx9"
    "l68qeY8mw+ynxh25RPe6SMG0eCE/ZZfIeWIcKUKwWujYBE5O5TDezLnO93BLCm/0UDq5vCNzk8GvwxzLJWqmkGXWrIGm0xCO"
    "QC+0PBcJkwpIX9h8lccAF9dsO2A5NjFgNyxCzx2YJH4+/eXoy9G0Mk4Sr/9+dL3Cfk+KZ5/Y4q5Ia/P25Xh2UjwdCJa8NIJB"
    "qjKsRzhzIph8mfR3H45aTAdaFaJeSJ1ho+RZ08TiZTsFI7+XgsPgZLK1BoglqYTWkjTyXUg6fxJ9YoQTKltvsuX9T1txxxRM"
    "01hm796Jkxo/4NWTHPX19f2P18UyN6K78mFz7/aDC/XhnJR7CY92AZISw9QZ6ZsVqTfa/UnFfLBfnITk2XqSPp4YjYS4TKmB"
    "58OJOg5KWNGFUdJHkehP4b8KhKUSTSKnY2AHlzKkBSCOUzL6fH92lG3i+xMR/vPttzH6uKEHk1HGiIIDf1vDuKLhwBGL59Xx"
    "bMzwrYggUcIEHqObwy6HxESx5m20mfaUvFeq6nzXkAK9hZzL+sqp68FxjSdNSbPqO9hyz5awAFomGE/gpD99YdPqjgBNib8d"
    "oGwYw7frdLGbiuB04/a8dLqZNPPdvFLN66Rjib/JNST+DrQUEnHR2ae9F9glAz7v9U0NAhRpJSYoPm6RGHpC4HL8+/7Z6QmH"
    "VbGHt+qjbw4rum7VBPwgv/HBTivW8J12tUc8crnY0yeiBUyRQN7MWQto/FJ1VBLtUkEhhhfkFO46wjAC5L7UQPfwhgJf5RAH"
    "BnlmH+axlZSEYE041YBeyP90KGA1ciFTlgeIOpVKbZNVcVMIxT2JZAJDpJVTqmFQFpA94Ugn+NFQRTklHR6fcmRkGjkQ+Xm9"
    "j8xJcHlAR8ayxMwc6SwEKav5EGk34A3WIS8jngUQvaHLrq7HgSyHh86hIquswti8ppuKc1yocwvmou6ix194PBJCHdDB81rh"
    "oscROFiCg1gmeonnAtYmQeEaO4aq6tfRBJN/i1p19Ld62TNJqfCNPyeNPGUyENN41q8VDEKkzJUuLDZuFRseoTR2UvfiMzwQ"
    "tHv7OdxcuizXOyEVM/CAfhb1O+iAKK8qEQSq7WnjeKi/l3f3j1AQJGy5G1GvZpwIi9Y1HzBXRQIda3clYQAJnMgEz17y7eLQ"
    "3vMdvEICQsstbI3SbWfnlvBEAYG/yypMYxQ6IhSFBQd4tsnXlO0EohdSAPBCHGaYVGtpMlL+pxS2iQfFDa6MLKh2DlkhDUIH"
    "tMz0Lh8mICgoe66p8pcAJPzzQs6rgQHA9f7/kilnjXcqv6QTEXCAQ05HBQJ7Yuhg2oPDgQOC03HAY18ABi0KloK3DhstrMfd"
    "30xCIThrjDowTnt+WpLf+oRshDnQA8yBA2nCxybNSZMQoHJBgisxc/TtZZtLuZV9mdINOx2KHVP1Flk7WuTzwy8sKwRA80ea"
    "t1c8Qsm7sDp7F30n8/EoCF+vCpx19S50B+/CgfDRI5/K41IKQJvPr0CLLSvwyMENKEooyG+Xh2qHlBcLB4XJ4HeDH4DKfU+8"
    "r/GkRWtAnOnL7DoTMXB6GxI2tbeVYxvG5HTLbotR0QIv2dEQwGgLVwQrJA98KldsrkGzYTEiixSicmnhJAq9DJbjFsi6yrAo"
    "Zq5w0u/p1jUQD4TxYQcw8EeC83ZJ3E2C+7BTTL4qp0oNxEafMgz0gz5TXEKFjkj8BGvFPvEVc74Mo4sDT48oW/Gvxbat9efA"
    "vfF56BQuSQaivkCO2aiQnlqJ1zrXoNCZI5ILGqb8MCiCMB9x316iqEP+YqK1mVswEzJkJmIDTBS6xoKVTwMwm5QCExofm1ZB"
    "KhFqBy5InkOvOP6aIm28+CGaRTsLltK0V+ycjVItzBnZjWoUWHchz9w1YHTqncpWimrkDLqBUbYqEkLULa6NgJzKrUZ1UVOp"
    "OGtSxBiLtyBgCO18GjW4f5v6yc1mUmw4AXicHqKixHGBC9AxmocJc5wWkj6X+nnsLcaYNgtbQ4LX/bDKujwAlBBB3Ow9e7En"
    "V81SmZ38/+q8LO10cjlQa3FOG/p7YN4NaHyw7QLwgl1l67Uo1AgeXQjpZrAlFZNXk7TNu/Cfm0xsWTngYZu2hqr1LRKSEK1r"
    "314aIm4QOmrTNkBF0yINHCt1dZit2DUcOvvqYHKBU8j3TKTuIqKNMH+IzK3nwAVlWmOYt4oHZK4xI0sjs/dF6OBt85/D28b5"
    "ZT9iMXywBuT9/zCeKdXwtx6UDwTFGAwJVCohZxV3IhT6oAxy3JIbvSa+nOWa7uPsXDaquLzP7xwCr9ZxiWvFFU/OOVeOcz5j"
    "sjWOAE4lXSxgWR/PzJ6vMR8u2knsEbwsgeuLdUkng2Y4+icxfTCoU523NBguqZeXYNZrLh0tkyq/zZZAqBwvENr98H6nZNMo"
    "uJ3cLQMiH543Sjb5Tv529zvGj+HeYSS1cLCjgnyToUMhjZQ6Pj8crrA9AoQt9nmdz4CNusxKPvBN8woo6iC+tmw5H20r3pRr"
    "7FCibUZsIdc4jOroiJxe9eWS5zUUgRJ725SowarnNBiNAvl9deBsjEbHoI3POHhS/OhlHQhRmtCJsGDblRUb+DEfpY5MvlYs"
    "ziBzrJHNSG+mmWgiLdIiHmHWnL2XgtlJYTQwEW5RwbCZeUZzBRdixs/k1GUyNWCOjyCpqjq+9rGEYiEVZbrITiG6BY5Fmin3"
    "pRIRyzhkXbj1HBgkBzVWFE2j+bfDLluYt453EUxY/WJ1KLXA/fflmjJJqQTb2gZRbXcEk6TNG3e1OyDWfh2m94pX7GE8OZyV"
    "NDzWVM35MfHlg4IK0q4hQUDILSjcnCcwFwy6BKgrccQZfBs9XYDQp8YX6RyLRvSUhC5hElmooPhM3E8pytBnJF9+ypRpsBy3"
    "mkRa167AKtwBAaFr5aH9anu3FPRIwDieBclfZQ+Ny+uhA8WDp6X4SeOOGCD/s0dl0cGlPWFrGqipLxgpEYEpqMUO8EHIKxNh"
    "IHfWLQrfXG8PKIXH7uJNTR/vEZCVAvBdmOx2N7zA6ZAbmAjvrI+e435/HRLR0zdtbYDeDS9bJVLeDSSc/IQnaYEJXGIaFPmh"
    "QnbNoyeO+D+Tz1WcCl1l0w/YLe8EGkf3AmWVTF/PLpCfB2gfvdBjfiG0TCzXxUZsqHQI6lLW/dbM8CUdGbYT3qcacuSltffy"
    "MRv5jIKoH+06LPaGr25wDjXvIRLHL1WSR8j2r0nkxW+O39liAir+wO5rAQuF7sKaw7VIR+9qBaxlT2sAWWrrBSkWLnpb8pIP"
    "c1Hsy5+LfqTHa2/tPBBfCWKbGA+Dpulp5Ef21orRaO3NBKxJcQMk32QxMlorRg50n79g7e019nsHETiZkxQPhwlvbrZmAlzb"
    "Scxv9EjcPaVb3iHjTQwnMUv4IcCWrW6C5MMJpjsxnLic7+kLpryPq5eIo7Cnpp1Ff4awoDx5L+oJgAKGpTZobCH4CLM2k5no"
    "OiVLz0VWKRClg2o9h5YU9g9BiD0vdCkRuUiJxTNIidMeGRGWCVy8ER6Xt3SVeuyhIaagW/pabaHGoRMmtUOneSFURIiA21PY"
    "zvkUUTERlYOsOGjSRIXOOwJGFbFD1nCKHp8rV5Ch9Br3UPyGmjCMZwoaLI5qVogQGD+sE31qxmiVefDQvBfXuQKmB6gzayj4"
    "WGSLholuUGg48C1BjX6gDgziv2RNQyTqC2/Bax+33dzBtJAJKApZ5Dzk68VMNZxIXgBhmCatKwHZSyImvLI5ArqFRVS4bAl0"
    "XeNMHBRO/9HchQ/J95XF7y7vf90pyeJ4iMLKZj9Qhof3DdWelQ0YKbEcw3pQ2wV97k1Xd6yefolJEFpYo5lGySjEJFsJ1KW+"
    "Mvr3CRod7u+JNkFq6z8pXyY8LjAlXBwFoT0VLWyR6ym3zEDX+k3ycNlUNQN4m7Xm4Hl2usX8DKwkaxu1zW3U+kbfF1pfm0Vb"
    "DujOWx8AIfw6u6pyrGACKhxF4CDE89u/LTt1kTDZFdO0M1kWGcG6cFIzAdwBWFF8IJJ6hL2fKeY5ERoXtaucOojaG5ky9ZHc"
    "ETBZygqa711qr40UuKgw0ZPy08i6z1gFB7q5Yp7JLqfgH2ahhFhQAMyArTAOoli/qXDYYlfroO5EhYHZWecNNRcLr4dkV5CN"
    "NVBL4+saqtQlItkcJYy18krfElyse4mFDOSdgWDWvhN22ylTftXuQswlwhc0LoiZS9b0hS1tqxdJfJVHz2TCYcR4VcneEYae"
    "OzBlfB5q05F51QEqHK8joVkf56OJpBhZ2I0tAWKhKlGhXYkKe1J93E2uBhvD4FLx4NSpcCjfZ1Saz3W5d4u3w4tUBn33R/ha"
    "nowGY8RP6Kz97V+xwMFv/8qwwtfZenlYN02vG+TFSNxuWD6X/lzYPLTkLSWj2Err0KD1KHCaJa0CYhUcnaShC0mDF0fS4Ldf"
    "+sKo2hI3KyyGDds+dFPFGjQNgx5SWuBw21JCYGmoB0jDwIGG4Yuj4V//5X9hWfy3bE4ueC2r7P7nz6RxkO1DDScike+K6duo"
    "/PtBW+Usimz7E757K8wHuzTv5kCMB6vtUxUge/TqCRDBzIHG0UukcRCg1Q9/RAu38xcXf4nVJ+Vg1NHhRKQZjOCoEpVWHRvd"
    "PJfeE3gQoBEAV7wYXHrLmCw2Im38VKQ9rZdqAWIKkwHeiGsob6Xv1/1EXpCcS317OKTjY/Tr1INYdXS3+dHoC5GbUNw6nEvX"
    "kwjf4F61V1x2O9cbbI0oSS+emtrmhCFfqtko1OzAWMlznhlHZmb/9gvX2hFv2VXU0vZwWRpF6NVbMdRinq5RoAJNEtdo/hLV"
    "chguuN3hP/Kjm9PzwYhQ9jaPwon8n8QB85fHAXD0whZqipQ6qgYwBBZ7wWt3olQVtbCw99lwUARqIBbVTp2eb5Bh9MRwIKBz"
    "Ae1HFRDHXrsiCZEcYXzR0C/lHaiE/tbin4pEFq33EuftOxaooQBo4E65VRcvmNLhX//5f08cy9yRExE7RsqTMax9FQtY9UXj"
    "uFBVvaBpepBaC6Kn/kBVO2eYdPlZhcZqeXrp89QfprBT0Pr8DmM/WYB2An+EyXgdHgejD1So1KGWpaCZaWGtAVmLUvgLZyXe"
    "ClG30BRY7FGt18LyFw7k9V8eeYO//vP/cd230Ne62C6bLSSaQhn/sLdjd45OleOgdu/G7k7sYjECyS/rdMZdQTGE2E0SC82n"
    "GSfLPNWE5eyj2oFpS0Ki2vOikIg11cQsfTCIDr4+iIi+61UUWM7QM02GSiUQZ4Qvb9ujbtMUB3S31y16WCcHLJKO8m5yupuV"
    "raldzOxsJrqeiO5Au0dwqj2o3IXSDjdXzzNsdi+sTnAvkuFJRglXHEq1UMWLPVYqVsLAG53U702r9mb4Vi2ekx7LeCRcUfFN"
    "d6uynhDTFeKgXvvP7mQ7+uyOZG21MeYwjoGl164MKbspEDP2VPBzsIZbGMZe1WbnK/aNqtQHdSPdwo+iwCQGpy3US54wSjWl"
    "DJIpe++LMiS25QbD5yV6ivA6tawzpV6ylnkBO7yZLTJUMCOMhqgmEgLDk/MdloUot0vzCEGVaGoXvlPzgtgpUCNLyhEpVhxP"
    "DXk8jR24JHh8LjltPvAU85H8KRcKU7u84ruZV0JDespSwo9g30xTq2sEnouJ0Rc9tbq94DHfUpYFprtETDBsKYqp7v+Snweo"
    "Gb4wasZTynmfhg7UhLc8LPB6KzJ9puSEmFoNZDEtfIlBqam1Hww85mnJgdJ0ZBMIWz5qqneBkZ8HKBm9MErOpz7ddmWPcHV9"
    "8bjHZjE/LaUMnFIp1Klvv7Qr58bv8Xyc2sUwPudJ2gNS36AWYPyJa8jl+sEAbeMXJJnDqWhsFbl6PhqxVvfCwCnx7lTYNMTR"
    "7nQpJZvqim2KOL3Aq1My9mfdx8lQiw4ndPTI4zAirCtb92Q78ELywvZ5NF2Qvsoe2WIZxXLrNcwN64ZMmoNX4B7dZXj/a7oQ"
    "lzjCafBG4ZC/vwRDg6LHfjD103dCnWrKQAnNCBGypaxMQ1txn4ejh7M6I8amtkz1qi7y8wDLzE/JMg/Np78uPn9GUSmam1Bz"
    "rxTXTgBvS8FCcv1BKU8JGfigEEO5qfZdTKT/E/R916ver2sQdDlD9mvFsUX1gTXxxAhZhSlDKqRTzByg7GwCvJU6XlfYpKSg"
    "xaSghXY9EB6rqHWsP1fwuuAxAfNSpwNdEUAuWmL66Aw0rhMFpcB79Y0MyMi+//GqEBd2vgKe+ZR56/z+Z9hOYNJQ4w4aoBUp"
    "bIY0L39wcdeZrFYXQ8UiNFORFh1Es25pYdIdkKRR2nYY59w+DXJ9Tj1HTJR6wwwOPY4gDRF3jxDxzOL55NSR8fRp0GtafKOJ"
    "d9yt97/S4geYVIybNhGT4G6ktJM+iyNUjIHYbnHEpsXxAAz0IhQGbN6kTAyT0kFBMeKwPbGcR9BYj4zTTUXEeuqmsyp+GOob"
    "orhysI4FTfVm+lpcU3lDRr1WPk4vSCl6fOVYklI2MbugAbg7p9aoHjzu02GPQ48YZAxi7LEx1eN/8vMA0/gv7bRAsYs13B15"
    "RjQDVpRDGC+UQ7Ab31A+s6ZeDJAiuqABtH8D+9nRyyNu6DjwhI4If5AExkHiwhPByQ+SUzQnqeueJjO1EOc78g/ANb7lvsBu"
    "B9tsLziF7UZiefehGkgg5X6iKiBJI/t9FBN+UKfSsZWS2QYlxyBCPU97UOipl8w1LTHvUhn9D4h3wufxmFgrazc7SF+1xZSt"
    "dQOCfnP/I3T8KhshLzrPedQ7lZYQnftWr0jrwNBOf7lhIW9gIQ5xFHdWW6UTdnRERveJjEGDt1yMK1dTB83Tj57NWuF6LEBl"
    "H7Tr4IY0uflh3fbZZ4ryZZ/vfwXnQX2X/JoaG2XkSkK4cmuJ8e3F8awep/bHc2qEq+xTEgIBmQvRm3pHJtOIk0Io/el21RR6"
    "A2NzUmiMWL0jI51+zZ2HK5LZeSqof3SfVHJBriOTXNHieUzDTeE1FxkVvzhldw7KLnqrw+Mip5V6919RNS+oelckmyR1PXHh"
    "NBTuLixGDesErO9Zs039UdFSF3R0z5sdEVab1XXZoUyIyoyi96ayVUusBP6DlSXmJ3K5kdaCLiUoO77OqMxEFMz0ku1w9mAu"
    "YPyKWhAXy+KaqlAcsGxAtc1R3cEaKsVNqdWu0LSZeuCWTZohAMqim77T2USr0ohYwr7Hp5TqYK/TN4t5dcURJN0X6QDjEyti"
    "VjHxZ7EDg7hdCX4Qb4zt64rFKOj3iFoUUDLiQI0BVgfsgYIrlSpdgaC4SVl3X6cvF4nWLPMf0Cfb/gLqWQe9IzH1xysv94cb"
    "N1dbPeZCzv9eIoGXemYza+qNOqwNyixLqPVwIUZRzSQ5DVBbrV2hBv8yBf8LepGCP6JExOyV/a7rqz5teejX6MrT+N9BKZsj"
    "fgF/efbVyNuzlesN6ac47lDbQr8weP7Lj6g2ISev6PTDkijoC4ZfSH1/cjoYQYcodqicoUhYCorO407z1zpp8Bp7yJeC5sNc"
    "TBFBCftCTiuOGSJ8rLNxU1ZHHYbpfgJoPcF7MZDqQzLNrxKu+dVI3NR4w0is2IAn0xsrGe6NRawWnRGroVy8/wV077ItEOWn"
    "M6VrUgZ6ilEnii7MgFhCVsSv9XJQYqSoK+bEX2IqBHtBsImEIUqxJLTxFhzdEpFVKUbAFTIKBTAVLP25xKqPk3qx0LioFz4r"
    "dDUkWnk7VMySOCY+K3lb62NbedZkkOipO5Kg/itsrY+iF9cGMAHrGx1FnGzVnUpioDbOpUc1tdNtRQ2Vi6JStJgWLoRMpGpn"
    "faImiltRE8VWYQmPe5pXuyEzJPFMNNjaubEu8SIXJS45KVM9zOaTWxGXyp/RPkzjGe9ZwncUnXgl8xvEcmsJVTAN+bip5YhV"
    "/07jjso9ND31TNMnZm8B6vmpqQth5mdKGHFABrM+wgwZK1pFv6CW19QuJrRnuM265HEAQjFjc3rWf2JktLkI8fT5jmS+C5ow"
    "duSxrHZBFUcrluve0I8XpSbAxQILSEuHOubf/OevSKT7i1cP1hlhrguJgSDBgjqCzCy9iOTrqj4mBr6lARiXWTCl9uiLYSWR"
    "R0atq+eIBhsgWtgL69FzB4ZaPKPTheGoViyjk+q6FObysqxgfUAVIr9cRK5HOAxh0a4OmJfCyHg4g0DMwTr32s96S+NmjClI"
    "RV5iLARpGjlK9TRqxWkavbL3vHrV12PZGaEhyW6iwrfCemW0wuoNHCEnOd4ifxr50WUFCA/A78QPqA5RFHK4k6JQnBKxvOHM"
    "kd6h4jAI7S6PcPaKbYRoAh2iqQmOFzevRssbx7vjTyZvQAwDwTLoCoUkBOGRY+Ifpn/CUsIhIW5/wVKAcZBYXBEfD7R7jCMC"
    "swT3pZOAke9eyOnfS3DCdu9PtaaFh8QBcPbWngahlte+KD929Kn5in7u280EP37VJ5r6fwaTou3yA4aY1USdtVsN08J30GD9"
    "4Dw1WGnh92qwQ7rlLNZuwNYqxYwcBYwnq6PBOgAR113N6XmXlOGJciBSeHYa7B6vMFNYXqqwVNK49dkL91THG4pnbadmNPG7"
    "d//TaBenhCncP3SUJw4+zmbce/G+6HA54/pqjnRtMjixvk0Wm56Om1bllZ478FJ0dry0OeDxeF00Odbk2v2UXWLv7H0uSu0B"
    "j1yuZcsjcRKTIZS+0lKelEEQgV1V9z+Ok1BB65ieibqfnBkkh3iZKHlQ41ZP8darS3zPGdaiLxSEnUSPFT/DMhqHGV8Ov4ue"
    "UhPfjdfi0xtKD8+2VP0kgfBgRXGPnySY9YoAf6JdzJd+QBHXcGsZ7wqEQsfG9CdqFZ8vDyNyCeDtjM0uSU8U8ICLKngnXeqs"
    "daEJ6j1TX9xBjTXQ/R+YoSReZTb+d2tk5zpOKn4J7Y5bTMePLS3dxJnt4/3MbHNJ46jzxC29S6GonqYWOqmdoGuywgEuv4VH"
    "NbIgHnBRB53Jf7rkImWlwtu2ulufU0JZszoLUKGrmARV8ZArcoOvLDgPg212fNecl6NNM7lriRuiTnjyHfooJApu6xZiPR1V"
    "BXt3uDu4rYv7cmahpN6co9mcaR/DgrKZbg2QPgLq+8tle0Vnv71uvakoZRs43TRQ9wEKqsjYYyL7OmIbXNJhlvalaXYh1PtM"
    "m7uPTiP7VRKl4sen1LFXRBRqxbeyoVbIkatWDFOT+2NamT7/SyzIw9LNINsgJNs+a2D00W9UFXaiXvIiqBe9RvIFb2SDgdEC"
    "LcLFe0/JyrbuWyMPSmdIqr3xSOfk/EXQb45yjlphkvrGCbuvrqrDdVPf9FYYZk0zXprmvVBrfOygac9ojuh7SN2my1RgKmdo"
    "49HOYs9Vnb6j0dAzmfsQ6D189bPXgfjp09ghI33ozQ0JxRoBQxtWJlENEigZbgRkj7RHAiRHUtc4N/d2ItWXILK2T9CUGN9+"
    "rzNwNE1URNStr6DQo0H5PXc3jaubQW+8jFhjccpz4cFc8e2h2EmfFSiX93+B3w5ZUKKVJVYGuJNLUbeBQ3sWGu/54l3NcSVG"
    "i7lgKMMskNr3lUw5zXEH46B3noIGKd4Ku7REFSbm61vRTw2OLL+9v9Iq7KFv6S7sYROVVZe8gvp48/u2bl9saW0vsDRUBbff"
    "gYOO+AWa/eyKO8ukyg9oWTUcPsacYrvPL8NQ38ca2kHTIKfHWgu6vogpCcCY69VA0iKUR0HMSSo/Zmy2HoCdvg0GqD6Z5Mej"
    "AmVETv9MyQkSBlrVQP5lRvvjjm6UEF+/pw+RKY9ESBIE2e2HfpMB1QRSAZtf5cnrggXej4KrlmPMde81HodUYSbH0AdeNkD8"
    "wjeO+ukxaPWZ9j0InVSZ9YPz55ygYR2stY9/kEZ7BOvs9sX+IBsk5Vrba9NBRCc5Xf/3VZW3UVMvpFpNLyhFR8NGp+H0HnkI"
    "RByF4zeOKvPRPwencf4hNiZ1+AknVbqdQszP65lK6IfRSkZDgiq6fTq/Ig/r6RyLfvToirEr9f4T3pqle05ZU19NJgls1bt7"
    "TZ9oNSFAJhNQf2kxB68JKy/V427BOm3NGNpfi4EkI7nRwoAJ09IXtgyAkRj0bHIBm29wYY/303MHzojPfmOjt4HUcWZXw39p"
    "NW8x4bfEzj+4z6g7OWUA43lcimlQi701XNCuG3o0mNoPfcK9fJWXI+Kw8PYm3+OHLtUWp7t3BqoFXA4qPUol3l5B7LyCQv+U"
    "nuUhfHH5DJ5eldT+sH0GV0o/lZoSAYkc9z9Wecbf26avpZ8FIcLBACCpVqUAAT5/Ma/Z3iClzWbtV5HUSNW9DLqXt93htu0N"
    "JES2h4UdbBvySxy4wUV7fJmMAGfJ/c/0Ds8N7Tt20hAZXguQbwWYN1olffxatPElNzsV95jVkY/uTbGZRLrDHkchIurm96DA"
    "3hJT8Wj1uJkDv4SPzS9jrx+CI6+QC/WODDf5YUe8s8X6kFe0N7BR31z+rSoFxGKYvw91A/KqHv6BLyjSACBUC6rjIYjWoiKd"
    "cXglVEJUjhO6J/o6ok6uc1Lp6YlPGReJ1Y2RxHKebkGR49ChU2YIETbJVMWmzTWNHfgneo7zhssybdenTijNPOQzSg6eeZgz"
    "Jnclcg9+m633h4rO8UT7+gFSSMECxr4m+CgMCNIb0c6lZZ7XPiXoJW/qvilJ00GFuzan/w43EeWCFE7Qjw5/76EHJ+fbEMRP"
    "8bnwk3FNzjzPmy5Hy8OdVP3o4FloKzFCUqltk+gKEc5pNi5e1E24umyh8++wQLLDa/oTS0gsxbvgWjq76CnJk2itRyZ6qjtn"
    "ef9ztcR9Ax/hVUg7jnVmh8JpWHLy//1PqufEOhu+YaaF5YehsOI0OXTpa0mOs+E3oqtd/YmqAM3TiVWu4PNjzgceHRw2hAjb"
    "LKQfm9bMgfd6o3nEMvPnUlX6uAX/V26vwO23EksJtn0BS7n9iCUS0P/W8M38KL75Vgew7sxfa5MK2UyVtqYYtnuIwond6qEv"
    "eg6RoxBpVNouCqwya8Wj1Wrxa4cDJj1PQaJtOkaGNPaRm8KhnuQ0SVd4ODX0G2f0slDHtfPrN3kHq7kTqRfP5gHhLuwjeIvd"
    "G9e/FmOPc09XNSmwb66GUGRP7P6I643YU7ju1A4QDWZf/bp+fpg52CROofQnExRGSKxcU8OMeg9d0uYHO6QqkADEJpGjCmmZ"
    "C4CImX77xVQdozr7NbHu/yQyBPX9r5fA7l26D0BtFEgVHmt39gBtzVCHI8Ap3P5kJP+dasVjRdDdAa5Q0yU0sOeF6Skbu9Z+"
    "DbLReV9GyXu9dGA6oEGnxREuilHwWq/EI/og/OCciE9+iOyaqqoILwQa2zvpgrgz/AwdN0R0OjeERALtfWHwb7I7kDtvhUsE"
    "kxErYea/aV0BqmfiNfXNTpR3cDun9R9cV/fj3BNHIUsei2E0+ViZk9sicGDB8Fy0ze9zj1J0oV4k+D+zilbX4oinek8dNzxd"
    "How4T7yqoawkALXI95bvYSXepQIzVG4mpyBZ45O4cPPMBzMK5zd+8ZDki/XGQZJ6jBFzHCqtb55Dgj3YUptzPkkduCo6u4Ot"
    "8SKqKp845hD2XnAGZgXZD7JGqGG554Kabh/YqP13osfhPgcfuLja+V0BQOEHZpA20Q4XXgn4960X0d8LpBDmIGtFm79VQeJJ"
    "Aw22yYdhFX9xjyl8EsRwyh6U2Ku6vGnsIivj8/HXYzLED1LnNdxE9YH0DlipPoUotE67dd+JDUql1FVXhuj8bpNp52HHbJFi"
    "RjWcYtGZTBb8d3TPu+nRozDCuTq4OPvmbTr2kGsephxz4RsQWIE9AIm2f0QegaoA+O7Kwk7+7LTlwpXCZqIaGeV/Ns0GMNVI"
    "NGcQ+StgG02gM0M4gayVaILJ39Fk1i2RVxJ8JNXuYE7hkBW7hbwlAQovAE0wPIfA8F9RY3sS96QcqY0nAAM5GQ6i0kCUs+sH"
    "1jOMvuDTYl3wsiUiuWLEevhqtFqnXv1kgAWdcmNPyX0jKntflutd1tZajvEBLHJV/pDt8BzLPh/WVJ0kgPr01Ur2NgD1CRpz"
    "V5kMVsIQVEeg30dV115ei5PAU38LLju1Z80zmvcD26W6KSFDyXM4jtDCw+xtk8mvFVb91gBFmwYz8D5CL/gqu1xTmwssUS4e"
    "X5e7Ah9ejOtl7YqYFJMPQemEHa+JC8NH5MKTlcRSTy3qPyLulWF3XyrjfFffxOo5u+D/tDa4kpSS4uMZEUR1rRLHI6uZI47q"
    "Uge+VWn3mTvCA4g4nFF2FFg/pH6F2E8dmCI6W6ZQpBmsyqZcZVRcC06qHw6oj0OSeUz/mwNTLHgbjkaqmf14AtENth0Vfyz2"
    "ucITWAwNStBm1OcAPdk3aCVSfw7U72gcHnk7PPJu8s/vPCgv+DPNuW5h2VXzjk3nghoOOhVSrHI+Rh8ntomfnG2OdF7j78de"
    "VVuDe6J3mPLrA/v4eD8uCHsyrUBfRJcXLiIaj14JYnArc7nQTxrX+1QXRzVclcBLwnKtdtVqcr6wSpo9sBFb+jc54yQl0XHY"
    "8FUYx90tQ55JzkELsrHLV9DMOtt+Fl6nFQYFUWXOCxTXVbYb0mhKuj1OgyOGi/5jhh5UWEe4yidrfAhRT9TC/g1bciI2ikG/"
    "YuPbmxv7ZnPj8WAZtcXv6WjsGx2N/aGOxsQM82exyo7hCDxAt9A7xEmxzdZXGXFNDyMENopc0CgPq0VCdEW9C2+jk37Ad6nl"
    "1uPYFRm6D38cGifraExMkz71CTK2KSV5hmR9xxms+/JTtqMkcTySyaSqraddRjXuM/HSWOMJp+I1niONjnSq1fgb7lKMA6gX"
    "ulVqweOOwvMQzNw6Jys4sRXjprrwkp8HeG/x4qyn2lUCPpLZO3KO3IEOOJs4e35gnPSqYJ2CmbjHOXDNDN+oTZVoYr9+MOja"
    "YQAzt8tMkKz/ZtR1UaC5kZHxCELqZBYS0L4CD2ypK7loM0NZ0YAoz3uZxUhVMRRRTFH4pX2I0UytWwoEyGO25IHf7WMzCpDo"
    "mNKCOKEq6lbm4LkECcRCSYRsMfIOKQo/5LiJZSNWLAuANV1RYoCPyJQmI+RIW7SHRApYlz0mUOOGQSNUQ4FEG/AZGFRQNgkd"
    "ND40LBMmiHm892u18RTXkURcc8wLy9VVoTkhpowi3I/jSbUdP3jpNrZca1SAuHADLH5UW6qG10KzV023IGqr0nrV2Op11CZV"
    "sCFSQ/6cBgdK6jChn8zh4ofnajxj9ApSN66hFDdIb+i0gYEp8KoA7YHwkFJVol6XbykGdf/rWnR/WBbbZSk6HK9sbTiUpRYT"
    "YIqHfP3DsPuXxmBKhPDJ+6hNwP9nZtCgU6ixQQmXiDro0JVtm3orgPCe334crGUiGeisItui0CqzDrwUnbP0e6Az5vd/r/pj"
    "Ei77UOxgq28EfMqgh5JcAE5MhmSW1RMTm56YMQBZ0cP7YGLDBxO7+GD8+FzPlG5DT8guAYkFS4QcADmZbTdPWLKP97/uIQrX"
    "BCbXMDdNIIeM0YrkxJky74MNbaXfZjzVcnsG7NlYjsBUCbuRHR5vZJtYOdjXBj58gDI24pOMhPv//4ObqfNRAQA="
)

if not RUTA_DATOS.exists():
    RUTA_DATOS.parent.mkdir(parents=True, exist_ok=True)
    RUTA_DATOS.write_bytes(gzip.decompress(base64.b64decode(_BLOB)))
    print(f"Corpus escrito en {RUTA_DATOS} (copia embebida).")
else:
    print(f"Se usará el corpus existente en {RUTA_DATOS}.")

sha_real = hashlib.sha256(RUTA_DATOS.read_bytes()).hexdigest()
print("SHA-256:", sha_real[:16], "…")
print("Coincide con la versión embebida:", sha_real == SHA_ESPERADO)

### 4.2 · Carga y particiones

Las particiones vienen **fijadas en el archivo** (campo `split`), no se
calculan aquí. Es una decisión deliberada: si cada notebook hiciera su propio
`train_test_split`, cuatro arquitecturas estarían evaluándose sobre conjuntos
distintos y las métricas no serían comparables entre sí.

La partición es estratificada por categoría (12 entrenamiento + 3 validación
por clase), generada con semilla 42.

In [ ]:
import json
from collections import Counter

registros = [json.loads(l) for l in RUTA_DATOS.read_text(encoding="utf-8").splitlines()]

train = [r for r in registros if r["split"] == "train"]
val   = [r for r in registros if r["split"] == "validation"]
demo  = [r for r in registros if r["es_demo"]]

CATEGORIAS = [
    "suma", "resta", "multiplicacion", "division", "operaciones_combinadas",
    "potencias_raices", "fracciones", "porcentajes", "ecuaciones",
    "geometria", "estadistica_probabilidad",
]
CAT2ID = {c: i for i, c in enumerate(CATEGORIAS)}
ID2CAT = {i: c for c, i in CAT2ID.items()}

print(f"Total: {len(registros)}  |  train: {len(train)}  |  validación: {len(val)}")
print(f"Ejemplos de demostración (todos en validación): {[d['id'] for d in demo]}")
print()
print("Distribución por categoría (train / val):")
ctr, cva = Counter(r["categoria"] for r in train), Counter(r["categoria"] for r in val)
for c in CATEGORIAS:
    print(f"  {c:26s} {ctr[c]:3d} / {cva[c]:2d}")
print()
print("Ejemplo completo:")
print(json.dumps(train[0], ensure_ascii=False, indent=2))

Un ejemplo del corpus se ve así:

```
entrada : "María tiene 48 caramelos y quiere repartirlos por igual entre 6
           amigos. ¿Cuántos caramelos recibirá cada amigo?"
salida  : "Paso 1: Repartir en partes iguales es dividir.
           Paso 2: 48 ÷ 6 = 8.
           Respuesta final: 8 caramelos"
valor   : "8"
```

El campo `valor` es la clave de toda la evaluación automática: es la respuesta
en forma canónica (sin unidades ni texto). Comparar `valor` contra lo que el
modelo escribe después de `Respuesta final:` nos da una métrica objetiva de
**si el modelo resolvió bien el problema**, independiente de cómo lo redactó.

In [ ]:
import json
from pathlib import Path

DIR_RESULTADOS = Path("resultados")
DIR_RESULTADOS.mkdir(exist_ok=True)


def guardar_resultados(nombre, payload):
    """Persiste las métricas para que el notebook de comparación las agregue.

    Sin este paso, comparar las cuatro arquitecturas obligaría a re-ejecutar
    todo en una sola sesión. Con él, cada notebook se ejecuta cuando se pueda y
    la comparación se hace al final leyendo los JSON.
    """
    limpio = {}
    for k, v in payload.items():
        if isinstance(v, dict):
            limpio[k] = {kk: vv for kk, vv in v.items() if not kk.startswith("_")}
        elif not k.startswith("_"):
            limpio[k] = v
    ruta = DIR_RESULTADOS / f"{nombre}.json"
    ruta.write_text(json.dumps(limpio, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Métricas guardadas en {ruta}")
    return ruta

## 5 · Preprocesamiento

Para clasificación el preprocesamiento es notablemente más simple que para
generación: la entrada es el enunciado, la etiqueta es un entero de 0 a 10.

Un detalle que sí importa: **el modelo solo ve `entrada`, nunca `salida`**.
Sería trivial clasificar leyendo el procedimiento ("Dividimos 48 entre 6" dice
la categoría en voz alta), pero en producción el procedimiento todavía no
existe: es justo lo que queremos generar después. Entrenar con él sería una
fuga de información que inflaría las métricas y no funcionaría en el pipeline.

In [ ]:
from collections import Counter

X_train = [r["entrada"] for r in train]
y_train = [r["categoria_id"] for r in train]
X_val   = [r["entrada"] for r in val]
y_val   = [r["categoria_id"] for r in val]

print(f"Entrenamiento: {len(X_train)}  |  Validación: {len(X_val)}")
print(f"Clases: {len(CATEGORIAS)}")
print(f"Distribución en validación: {dict(Counter(y_val))}")
print()
for i in range(3):
    print(f"  [{CATEGORIAS[y_train[i]]}] {X_train[i]}")

## 6 · Tokenización

BETO usa **WordPiece**: parte las palabras en subunidades marcando con `##` las
continuaciones. Comparen mentalmente con lo que verán en los otros notebooks
(BPE en Qwen, SentencePiece en FLAN-T5); en el notebook 5 lo cuantificamos.

In [ ]:
import numpy as np
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODELO_ID)

longitudes, fertilidades = [], []
for r in registros:
    ids = tokenizer(r["entrada"])["input_ids"]
    longitudes.append(len(ids))
    fertilidades.append((len(ids) - 2) / len(r["entrada"].split()))

longitudes = np.array(longitudes)
print(f"Tokens por enunciado: min={longitudes.min()}  media={longitudes.mean():.1f}  "
      f"p95={np.percentile(longitudes, 95):.0f}  max={longitudes.max()}")
print(f"Ejemplos que excederían LONGITUD_MAX={LONGITUD_MAX}: {(longitudes > LONGITUD_MAX).sum()}")
print(f"Fertilidad (tokens/palabra): {np.mean(fertilidades):.3f}")
print(f"Vocabulario: {tokenizer.vocab_size}")

STATS_TOKENIZADOR = {
    "modelo": MODELO_ID,
    "tokenizador": tokenizer.__class__.__name__,
    "vocabulario": int(tokenizer.vocab_size),
    "fertilidad_media": float(np.mean(fertilidades)),
    "tokens_media": float(longitudes.mean()),
    "tokens_max": int(longitudes.max()),
}

In [ ]:
muestras = [
    "¿Cuál es el 25% de 420 estudiantes?",
    "Resuelve la siguiente operación: 864 ÷ 12.",
    "Calcula el área de un círculo de radio 5 usando π = 3.14.",
]
for m in muestras:
    piezas = tokenizer.tokenize(m)
    print(f"\n({len(piezas)} tokens) {m}")
    print("   " + " | ".join(piezas))

ids_unk = [i for m in muestras for i in tokenizer(m)["input_ids"] if i == tokenizer.unk_token_id]
print(f"\nTokens desconocidos: {len(ids_unk)}")

In [ ]:
from datasets import Dataset


def tokenizar(reg):
    tok = tokenizer(reg["entrada"], truncation=True, max_length=LONGITUD_MAX)
    tok["labels"] = reg["categoria_id"]
    return tok


ds_train = Dataset.from_list([tokenizar(r) for r in train])
ds_val   = Dataset.from_list([tokenizar(r) for r in val])
print(ds_train)
print("\nPrimer ejemplo decodificado:")
print(tokenizer.decode(ds_train[0]["input_ids"]))
print("Etiqueta:", CATEGORIAS[ds_train[0]["labels"]])

In [ ]:
from transformers import DataCollatorWithPadding

collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")
print("Collator con padding dinámico listo.")

### Métricas de clasificación

Cuatro métricas, y hay que tener claro qué mide cada una porque con 11 clases
balanceadas es fácil citar la equivocada:

| Métrica | Qué responde |
|---|---|
| **Accuracy** | ¿Qué proporción del total acertó? Con clases balanceadas es informativa; con clases desbalanceadas engaña. |
| **Precision** (por clase) | De todo lo que predije como `porcentajes`, ¿cuánto lo era de verdad? Penaliza los falsos positivos. |
| **Recall** (por clase) | De todos los `porcentajes` reales, ¿cuántos detecté? Penaliza los falsos negativos. |
| **F1-macro** | Media armónica de precision y recall, promediada **dando el mismo peso a cada clase**. Es la métrica principal de este notebook: si el modelo ignora una clase entera, el F1-macro se hunde aunque la accuracy apenas se mueva. |

Se reporta también el F1 ponderado por soporte, pero con nuestro conjunto
balanceado (3 por clase) coincide casi exactamente con la accuracy y aporta
poco.

In [ ]:
import numpy as np
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, precision_recall_fscore_support)


def metricas_clasificacion(y_true, y_pred):
    p_ma, r_ma, f1_ma, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0)
    p_pe, r_pe, f1_pe, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision_macro": float(p_ma),
        "recall_macro": float(r_ma),
        "f1_macro": float(f1_ma),
        "precision_ponderada": float(p_pe),
        "recall_ponderado": float(r_pe),
        "f1_ponderado": float(f1_pe),
    }


def compute_metrics(eval_pred):
    """Callback del Trainer: se ejecuta al final de cada época."""
    logits, labels = eval_pred
    return metricas_clasificacion(labels, np.argmax(logits, axis=-1))


def imprimir_metricas(nombre, m):
    print(f"{nombre:34s} acc={m['accuracy']:.3f}  F1-macro={m['f1_macro']:.3f}  "
          f"P-macro={m['precision_macro']:.3f}  R-macro={m['recall_macro']:.3f}")

## 7 · Baseline

Aquí la metodología se separa de la del notebook de Qwen, y conviene explicar
por qué.

Un modelo generativo sin entrenar **sí sabe hacer la tarea** (mal, pero la
hace): se le pide que resuelva un problema y produce texto. Un encoder sin
entrenar **no puede clasificar en absoluto**: su cabeza de clasificación acaba
de inicializarse con números aleatorios. Preguntarle es preguntarle a un dado
de 11 caras.

Por eso usamos tres referencias, no una:

1. **Clase mayoritaria** — el mínimo absoluto. Con 11 clases balanceadas está
   en 1/11 ≈ 9%. Cualquier sistema debe superarlo o no sirve para nada.
2. **TF-IDF + regresión logística** — el baseline honesto. Un modelo clásico,
   sin redes neuronales, entrenado en dos segundos. Si BERT no lo supera, no
   hay ninguna razón para pagar el costo de un transformer de 110M de
   parámetros. *Esta es la comparación que de verdad importa.*
3. **BETO con la cabeza sin entrenar** — se incluye para hacer visible que el
   preentrenamiento aporta representaciones, pero no la tarea.

In [ ]:
# Baseline 1: clase mayoritaria
from collections import Counter

clase_mayoritaria = Counter(y_train).most_common(1)[0][0]
pred_mayoritaria = [clase_mayoritaria] * len(y_val)
m_mayoritaria = metricas_clasificacion(y_val, pred_mayoritaria)
imprimir_metricas("1) Clase mayoritaria", m_mayoritaria)
print(f"   (predice siempre '{CATEGORIAS[clase_mayoritaria]}')")

In [ ]:
# Baseline 2: TF-IDF + regresión logística
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline

clasico = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, min_df=1),
    LogisticRegression(max_iter=2000, C=5.0),
)
clasico.fit(X_train, y_train)
pred_clasico = clasico.predict(X_val)
m_clasico = metricas_clasificacion(y_val, pred_clasico)
imprimir_metricas("2) TF-IDF + LogisticRegression", m_clasico)

# Con solo 3 ejemplos de validación por clase, una única medición es ruidosa.
# La validación cruzada sobre el corpus completo da una estimación mucho más
# estable de lo que el enfoque clásico puede lograr. Es barata: úsenla como
# referencia principal al discutir si BERT vale la pena.
X_todo = [r["entrada"] for r in registros]
y_todo = [r["categoria_id"] for r in registros]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
scores = cross_val_score(clasico, X_todo, y_todo, cv=cv, scoring="f1_macro")
print(f"   Validación cruzada 5-fold sobre los 165 ejemplos: "
      f"F1-macro = {scores.mean():.3f} ± {scores.std():.3f}")

In [ ]:
# Baseline 3: BETO con la cabeza de clasificación sin entrenar
import torch
from transformers import AutoModelForSequenceClassification

modelo = AutoModelForSequenceClassification.from_pretrained(
    MODELO_ID,
    num_labels=len(CATEGORIAS),
    id2label={i: c for i, c in enumerate(CATEGORIAS)},
    label2id={c: i for i, c in enumerate(CATEGORIAS)},
).to(DEVICE)


@torch.no_grad()
def predecir(modelo, textos, batch=16):
    modelo.eval()
    salida = []
    for i in range(0, len(textos), batch):
        lote = tokenizer(textos[i:i + batch], padding=True, truncation=True,
                         max_length=LONGITUD_MAX, return_tensors="pt").to(modelo.device)
        salida.extend(modelo(**lote).logits.argmax(-1).cpu().tolist())
    return salida


pred_sin_entrenar = predecir(modelo, X_val)
m_sin_entrenar = metricas_clasificacion(y_val, pred_sin_entrenar)
imprimir_metricas("3) BETO sin fine-tuning", m_sin_entrenar)
print("   (la cabeza está inicializada al azar: el resultado debe rondar 1/11 = 9%)")

## 8 · Configuración del fine-tuning

### Por qué fine-tuning completo y no LoRA

En el notebook de Qwen, LoRA es prácticamente obligatorio: 1.500 millones de
parámetros no caben en memoria con sus estados de optimizador. Aquí la
situación es distinta:

- BETO tiene 110M de parámetros. Entrenarlo entero en una T4 cabe de sobra.
- La cabeza de clasificación es **nueva** y hay que entrenarla sí o sí.
- Con tan pocos datos, permitir que se ajusten todas las capas suele dar mejor
  resultado en clasificación que restringirse a adaptadores de bajo rango.

Aplicar LoRA aquí sería usar la herramienta por costumbre y no por necesidad.
De todos modos dejamos la variante lista abajo, porque compararlas es una
observación interesante: es el mismo dilema, con distinta respuesta, que en el
notebook 1.

### Hiperparámetros

| Parámetro | Valor | Razón |
|---|---|---|
| `learning_rate` | 3e-5 | Rango estándar para fine-tuning completo de BERT (2e-5 a 5e-5). Dos órdenes de magnitud menor que el de LoRA: aquí se mueven los pesos preentrenados y hay que hacerlo con cuidado. |
| `num_train_epochs` | 15 | Con 132 ejemplos y lotes de 8, una época son ~17 pasos. Hacen falta bastantes épocas para que converja. |
| `batch_size` | 8 | Lotes pequeños dan más pasos de actualización, que es lo que escasea con un corpus mínimo. |
| `warmup_ratio` | 0.1 | Evita que los primeros pasos, con la cabeza aleatoria produciendo gradientes enormes, dañen las representaciones preentrenadas. |
| `metric_for_best_model` | `f1_macro` | No `eval_loss`: nos interesa la calidad de clasificación por clase, no la verosimilitud. |

In [ ]:
import inspect
from transformers import TrainingArguments


def construir_args(clase=TrainingArguments, **kwargs):
    """Crea TrainingArguments filtrando los parámetros que la versión instalada
    de `transformers` no reconoce.

    Motivo: entre versiones recientes `evaluation_strategy` pasó a llamarse
    `eval_strategy`. Sin esta capa, el notebook funciona hoy y falla el
    semestre que viene. Se prefiere `eval_strategy` y se traduce si hace falta.
    """
    admitidos = set(inspect.signature(clase.__init__).parameters)
    if "eval_strategy" in kwargs and "eval_strategy" not in admitidos:
        kwargs["evaluation_strategy"] = kwargs.pop("eval_strategy")
    descartados = [k for k in kwargs if k not in admitidos]
    for k in descartados:
        kwargs.pop(k)
    if descartados:
        print(f"Parámetros no soportados por esta versión, se omiten: {descartados}")
    return clase(**kwargs)

### Weights & Biases

W&B registra automáticamente la curva de pérdida, los hiperparámetros y el
consumo de GPU. Es lo que después nos permitirá comparar las cuatro
arquitecturas sobre los mismos ejes en lugar de sobre capturas de pantalla.

Convención de nombres del proyecto (idéntica en los cinco notebooks):

- **Proyecto:** `tutor-matematicas-arquitecturas`
- **Run:** `bert-clasificador-finetune`
- **Tags:** identifican arquitectura y fase, para poder filtrar en el panel

Si no quieren usar W&B, pongan `USAR_WANDB = False`: el notebook seguirá
funcionando y las métricas se guardarán igual en `resultados/`.

In [ ]:
import os

USAR_WANDB   = True          # ponlo en False para trabajar sin conexión a W&B
PROYECTO     = "tutor-matematicas-arquitecturas"
NOMBRE_RUN   = "bert-clasificador-finetune"
TAGS         = ["bert", "beto", "encoder-only", "clasificacion"]

if USAR_WANDB:
    import wandb
    wandb.login()            # pedirá la API key la primera vez
    os.environ["WANDB_PROJECT"] = PROYECTO
    os.environ["WANDB_LOG_MODEL"] = "false"
    REPORTAR_A = "wandb"
else:
    os.environ["WANDB_MODE"] = "disabled"
    REPORTAR_A = "none"

print(f"Registro de experimentos: {REPORTAR_A}  |  run: {NOMBRE_RUN}")

In [ ]:
from transformers import Trainer

USAR_LORA_EN_BERT = False    # ver la discusión de arriba

if USAR_LORA_EN_BERT:
    from peft import LoraConfig, get_peft_model
    config_lora = LoraConfig(
        r=8, lora_alpha=16,
        target_modules=["query", "value"],
        lora_dropout=0.05, bias="none",
        task_type="SEQ_CLS",
        modules_to_save=["classifier"],   # la cabeza nueva se entrena entera
    )
    modelo = get_peft_model(modelo, config_lora)
    modelo.print_trainable_parameters()

args = construir_args(
    output_dir=f"{DIR_CHECKPOINTS}/bert-clasificador",
    run_name=NOMBRE_RUN,

    num_train_epochs=15,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,

    learning_rate=3e-5,
    lr_scheduler_type="linear",
    warmup_ratio=0.1,
    weight_decay=0.01,

    fp16=(DEVICE == "cuda"),
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,

    report_to=REPORTAR_A,
    seed=SEMILLA,
)

trainer = Trainer(
    model=modelo,
    args=args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

n_entrenables = sum(p.numel() for p in modelo.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in modelo.parameters())
print(f"Parámetros entrenables: {n_entrenables/1e6:.1f} M de {n_total/1e6:.1f} M "
      f"({100*n_entrenables/n_total:.1f}%)")
print(f"Pasos por época: {len(ds_train) // args.per_device_train_batch_size}")

## 9 · Entrenamiento

Qué observar mientras corre:

- La `eval_loss` de un clasificador con 11 clases arranca cerca de
  `ln(11) ≈ 2.40`, que es la entropía de adivinar al azar. Ver ese número en
  la primera evaluación confirma que todo está bien conectado.
- El `f1_macro` suele dar un salto brusco en las primeras 3-4 épocas y luego
  aplanarse. Ese aplanamiento no significa que el entrenamiento falló:
  significa que el modelo ya extrajo lo que 132 ejemplos podían darle.
- Si `eval_loss` sube mientras `f1_macro` se mantiene, el modelo se está
  volviendo **más confiado** en sus errores. Es sobreajuste incipiente y por
  eso seleccionamos por F1 y no por pérdida.

In [ ]:
resultado_entrenamiento = trainer.train()
print()
print(f"Tiempo de entrenamiento: {resultado_entrenamiento.metrics['train_runtime']:.1f} s")

In [ ]:
import pandas as pd

historial = pd.DataFrame(trainer.state.log_history)
ev = historial.dropna(subset=["eval_f1_macro"])
print(ev[["epoch", "eval_loss", "eval_accuracy", "eval_f1_macro"]].to_string(index=False))

mejor = ev.sort_values("eval_f1_macro", ascending=False).iloc[0]
print(f"\nMejor época: {mejor['epoch']:.0f}  (F1-macro = {mejor['eval_f1_macro']:.3f})")

In [ ]:
import matplotlib.pyplot as plt

tr = historial.dropna(subset=["loss"])
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(tr["epoch"], tr["loss"], label="Entrenamiento", alpha=0.8)
axes[0].plot(ev["epoch"], ev["eval_loss"], label="Validación", marker="o")
axes[0].axhline(np.log(len(CATEGORIAS)), ls=":", c="red", lw=1, label="Azar (ln 11)")
axes[0].set_xlabel("Época"); axes[0].set_ylabel("Pérdida")
axes[0].set_title("Curvas de pérdida"); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(ev["epoch"], ev["eval_accuracy"], marker="o", label="Accuracy")
axes[1].plot(ev["epoch"], ev["eval_f1_macro"], marker="s", label="F1-macro")
axes[1].axhline(m_clasico["f1_macro"], ls="--", c="gray", lw=1,
                label="TF-IDF (F1-macro)")
axes[1].set_xlabel("Época"); axes[1].set_ylabel("Métrica"); axes[1].set_ylim(0, 1)
axes[1].set_title("Métricas de clasificación"); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

In [ ]:
trainer.save_model(DIR_MODELO)
tokenizer.save_pretrained(DIR_MODELO)
print(f"Clasificador guardado en {DIR_MODELO}/")
print("El notebook 3 (pipeline Qwen+BERT) cargará este modelo como etapa 1.")

## 10 · Integración con Weights & Biases

El `Trainer` ya envió a W&B, época por época, la pérdida y las cuatro métricas
de `compute_metrics`. Lo que añadimos aquí es lo que no se puede reconstruir a
partir de escalares:

- La **matriz de confusión** como objeto interactivo de W&B.
- La tabla de **errores individuales**: qué enunciado, qué predijo, qué era.
- Los baselines, para que en el panel aparezcan como líneas de referencia junto
  a la curva del modelo.

Nombres consistentes con los otros notebooks: proyecto
`tutor-matematicas-arquitecturas`, run `bert-clasificador-finetune`, tags
`bert` / `encoder-only` / `clasificacion`.

## 11 · Evaluación

In [ ]:
pred_finetuned = predecir(trainer.model, X_val)
m_finetuned = metricas_clasificacion(y_val, pred_finetuned)

print("Modelo con fine-tuning")
print("=" * 60)
for k, v in m_finetuned.items():
    print(f"  {k:22s}: {v:.4f}")

### Reporte por clase

El promedio macro esconde qué clases funcionan. Con 3 ejemplos de validación
por clase, cada acierto o fallo mueve el recall de esa clase en saltos de 0.33:
**no interpreten diferencias pequeñas entre clases como reales**. Lo que sí es
informativo es una clase con recall 0.

In [ ]:
print(classification_report(
    y_val, pred_finetuned,
    labels=list(range(len(CATEGORIAS))),
    target_names=CATEGORIAS,
    zero_division=0,
    digits=3,
))

### Matriz de confusión

Aquí es donde está la información realmente accionable. No miren solo la
diagonal: miren **qué se confunde con qué**. Las confusiones esperables por
construcción del esquema de etiquetado son:

- `operaciones_combinadas` ↔ `suma`/`resta`/`multiplicacion`/`division`: la
  frontera es "¿cuántas operaciones distintas hay?", una distinción que exige
  contar operadores, no reconocer vocabulario.
- `porcentajes` ↔ `multiplicacion`: calcular un porcentaje *es* una
  multiplicación; solo el símbolo `%` los separa.
- `potencias_raices` ↔ `geometria`: "área de un cuadrado" implica un cuadrado
  en ambos sentidos de la palabra.

Si aparecen estas confusiones, el modelo está fallando donde el esquema de
etiquetado es genuinamente difícil. Si aparecen otras (por ejemplo `geometria`
confundida con `fracciones`), el problema es falta de datos.

In [ ]:
cm = confusion_matrix(y_val, pred_finetuned, labels=list(range(len(CATEGORIAS))))

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(CATEGORIAS))); ax.set_yticks(range(len(CATEGORIAS)))
ax.set_xticklabels(CATEGORIAS, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(CATEGORIAS, fontsize=9)
ax.set_xlabel("Predicción"); ax.set_ylabel("Etiqueta real")
ax.set_title("Matriz de confusión · BETO fine-tuned (validación)")
for i in range(len(CATEGORIAS)):
    for j in range(len(CATEGORIAS)):
        if cm[i, j]:
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout(); plt.show()

In [ ]:
print("Errores de clasificación en validación:")
print("=" * 78)
errores = []
for r, p in zip(val, pred_finetuned):
    if p != r["categoria_id"]:
        errores.append({"id": r["id"], "entrada": r["entrada"],
                        "real": r["categoria"], "predicho": CATEGORIAS[p]})
        print(f"[{r['id']}] {r['entrada']}")
        print(f"    real: {r['categoria']}  ->  predicho: {CATEGORIAS[p]}")
print("=" * 78)
print(f"{len(errores)} errores de {len(val)} ejemplos")

## 12 · Comparación Baseline vs Fine-tuned

In [ ]:
filas = [
    ("Clase mayoritaria",           m_mayoritaria),
    ("TF-IDF + LogisticRegression", m_clasico),
    ("BETO sin fine-tuning",        m_sin_entrenar),
    ("BETO fine-tuned",             m_finetuned),
]

print(f"{'Modelo':32s} {'Accuracy':>9s} {'F1-macro':>9s} {'P-macro':>9s} {'R-macro':>9s}")
print("=" * 72)
for nombre, m in filas:
    print(f"{nombre:32s} {m['accuracy']:>9.3f} {m['f1_macro']:>9.3f} "
          f"{m['precision_macro']:>9.3f} {m['recall_macro']:>9.3f}")
print("=" * 72)
print(f"\nMejora de BETO sobre el baseline clásico (F1-macro): "
      f"{m_finetuned['f1_macro'] - m_clasico['f1_macro']:+.3f}")
print(f"Mejora de BETO sobre el azar (F1-macro): "
      f"{m_finetuned['f1_macro'] - m_mayoritaria['f1_macro']:+.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
nombres = [f[0] for f in filas]
x = np.arange(len(nombres)); ancho = 0.38
ax.bar(x - ancho/2, [f[1]["accuracy"] for f in filas], ancho, label="Accuracy")
ax.bar(x + ancho/2, [f[1]["f1_macro"] for f in filas], ancho, label="F1-macro")
ax.axhline(1/len(CATEGORIAS), ls=":", c="red", lw=1, label="Azar (1/11)")
ax.set_xticks(x); ax.set_xticklabels(nombres, rotation=15, ha="right", fontsize=9)
ax.set_ylim(0, 1.05); ax.set_ylabel("Métrica")
ax.set_title("Clasificación de problemas · baselines vs BETO fine-tuned")
ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
RESULTADOS_BERT = {
    "notebook": "S04_Lab_Fine_tuning_BERT",
    "arquitectura": "encoder-only",
    "modelo": MODELO_ID,
    "metodo": "LoRA" if USAR_LORA_EN_BERT else "fine-tuning completo",
    "n_train": len(train),
    "n_val": len(val),
    "n_clases": len(CATEGORIAS),
    "epocas": float(args.num_train_epochs),
    "learning_rate": args.learning_rate,
    "tiempo_entrenamiento_s": resultado_entrenamiento.metrics["train_runtime"],
    "eval_loss_final": float(ev.iloc[-1]["eval_loss"]),
    "mejor_epoca": float(mejor["epoch"]),
    "parametros_entrenables": int(n_entrenables),
    "parametros_totales": int(n_total),
    "baseline_mayoritaria": m_mayoritaria,
    "baseline_tfidf": m_clasico,
    "baseline_tfidf_cv_f1_macro": float(scores.mean()),
    "baseline_tfidf_cv_std": float(scores.std()),
    "baseline_sin_finetuning": m_sin_entrenar,
    "finetuned": m_finetuned,
    "matriz_confusion": cm.tolist(),
    "categorias": CATEGORIAS,
    "errores": errores,
    "tokenizador": STATS_TOKENIZADOR,
}

guardar_resultados("bert", RESULTADOS_BERT)

In [ ]:
if USAR_WANDB:
    import wandb

    if wandb.run is None:
        wandb.init(project=PROYECTO, name=NOMBRE_RUN, tags=TAGS, reinit=True)

    wandb.log({"evaluacion/matriz_confusion": wandb.plot.confusion_matrix(
        probs=None, y_true=y_val, preds=pred_finetuned, class_names=CATEGORIAS)})

    tabla = wandb.Table(columns=["id", "entrada", "real", "predicho", "acierto"])
    for r, p in zip(val, pred_finetuned):
        tabla.add_data(r["id"], r["entrada"], r["categoria"],
                       CATEGORIAS[p], p == r["categoria_id"])
    wandb.log({"evaluacion/predicciones": tabla})

    wandb.summary.update({
        "baseline/mayoritaria_f1_macro": m_mayoritaria["f1_macro"],
        "baseline/tfidf_f1_macro": m_clasico["f1_macro"],
        "baseline/tfidf_cv_f1_macro": float(scores.mean()),
        "baseline/sin_finetuning_f1_macro": m_sin_entrenar["f1_macro"],
        "finetuned/accuracy": m_finetuned["accuracy"],
        "finetuned/f1_macro": m_finetuned["f1_macro"],
        "delta/f1_macro_vs_tfidf": m_finetuned["f1_macro"] - m_clasico["f1_macro"],
    })
    wandb.finish()
    print("Registro en W&B completado.")
else:
    print("W&B desactivado; las métricas quedaron en resultados/bert.json")

## 13 · Discusión

**1. ¿BETO le ganó a TF-IDF?**
Es la pregunta central. Si la diferencia en F1-macro es pequeña (menos de ~0.10),
la conclusión honesta es que con 132 ejemplos el problema se resuelve casi
igual de bien con un modelo lineal sobre n-gramas. Y tiene sentido: nuestras
categorías se distinguen en buena medida por palabras clave ("porcentaje",
"área", "probabilidad"), que es exactamente lo que TF-IDF captura. La ventaja
de BERT debería aparecer en los casos donde el vocabulario no basta —"un
número más 15 es igual a 42" no contiene la palabra "ecuación"—. Vale la pena
revisar si esos casos concretos los acierta BERT y falla TF-IDF.

**2. ¿Qué confunde el modelo?**
Contrasten la matriz de confusión con las confusiones predichas más arriba. Si
coinciden, el límite es el esquema de etiquetado, no el modelo. Si no
coinciden, es falta de datos.

**3. ¿Es fiable esta medición?**
No del todo, y hay que decirlo. 33 ejemplos de validación, 3 por clase: un solo
acierto cambia la accuracy 3 puntos y el recall de una clase 33 puntos. La
validación cruzada del baseline clásico da una idea de la magnitud del ruido.
Para una medición sólida haría falta validación cruzada del modelo completo
(caro, 5 entrenamientos) o simplemente más datos.

**4. ¿Qué implica para el pipeline del notebook 3?**
El pipeline BERT→Qwen hereda estos errores. Si la accuracy es del 85%, uno de
cada siete problemas llegará a Qwen con la etiqueta equivocada y por tanto con
el prompt especializado equivocado. Esa **propagación de error** es la
desventaja estructural de cualquier sistema en cascada, y en el notebook 3 la
medimos explícitamente.

## 14 · Conclusiones

1. **El encoder es la herramienta correcta para esta tarea concreta.**
   Bidireccionalidad, inferencia barata y una salida cerrada y medible. Nada de
   eso lo ofrece un decoder.
2. **Pero no resuelve el problema del tutor.** Clasificar no es enseñar. BERT
   es un componente, no un producto.
3. **El baseline clásico es la referencia que importa.** Documenten la
   diferencia con TF-IDF: es lo que justifica (o no) el costo del transformer.
4. **La clasificación tiene sentido si algo la usa.** Su valor se demuestra en
   el notebook 3: si condicionar la generación por categoría no mejora nada,
   entonces esta etapa es complejidad sin retorno, y ese también sería un
   resultado válido del experimento.